# LangGraph Agentic RAG Evaluation Notebook

This notebook implements and compares 3 pipelines on a financial QA evaluation setup:

1. **Simple RAG** — dense retrieval only
2. **Advanced RAG** — contextual chunking + hybrid retrieval (BM25 + dense) + reranker
3. **LangGraph Agentic RAG** — see architecture diagram below

Dataset:
- **FinanceBench** (open-source subset, 150 questions) — financial QA over SEC 10-K/10-Q filings
- Pre-extracted page text from the FinanceBench repo (no PDF parsing required)
- Each question has a gold answer, gold evidence string, and source page reference

Evaluation metrics:
- Token F1, Exact Match (baseline)
- **Numerical Accuracy** — regex-based number extraction + tolerance match (key for financial answers)
- **LLM Judge**: correctness score + claims_covered fraction
- **Evidence Retrieval Hit** — strict: retrieved chunk must overlap with gold evidence string
- Pipeline intermediate metrics: context reject rate, critic repair rate, repair success

Notes:
- Designed for local Jupyter notebook
- Groq or Gemini for all LLM nodes (switch via `CONFIG["provider"]`)
- Pilot run defaults to 10 questions

---

### Agentic RAG Architecture (LangGraph)

```
User Question
      |
[AGENT] Query Rewriter
      |
[RAG]  Hybrid Retriever  <-----------------------------------------------+
      |                                                                   |
[AGENT] Context Evaluator                                                 |
      |                                                                   |
  relevant?                                                               |
  /        \                                                             |
yes    no (retries < max)                                                 |
  |        |                                                              |
  |   Increment Retry Counter --> [RAG] Hybrid Retriever (loops back up) |
  |                                                                       |
  v  (relevant OR retry cap hit: force through)                          |
[RAG]  Generator                                                          |
      |                                                                   |
[AGENT] Critic                                                            |
      |                                                                   |
  decision?    <- if is_repair_retrieval flag is set: skip to END        |
  /        \                                                             |
accept    repair                                                          |
  |         |                                                             |
 END    [AGENT] Repair Agent                                              |
              |                                                           |
      needs_new_retrieval?                                                |
      /           \                                                      |
    no            yes (count <= max_repair_retrievals)                   |
    |               |                                                     |
   END       [Mark Repair Retrieval] --------------------------------->---+
              (sets flag: skip critic on next pass)
```

**Key loop controls:**
- Context evaluator retries up to `max_context_retries` before forcing through to Generator
- Repair agent requests fresh retrieval up to `max_repair_retrievals` times
- After repair-triggered retrieval, Critic is **bypassed** to prevent infinite repair loops


In [1]:
# Install packages if needed
# Uncomment if running in a fresh environment
# !pip install \
# langgraph \
# langchain-core \
# langchain-groq \
# groq \
# langchain-google-genai \
# google-generativeai \
# datasets \
# requests \
# sentence-transformers \
# rank-bm25 \
# faiss-cpu \
# pandas \
# numpy \
# tqdm \
# scikit-learn


In [2]:
import os
import re
import time
import math
import json
import random
import textwrap
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, TypedDict, Literal, Tuple

import threading
from collections import deque

import requests

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

from pydantic import BaseModel, Field, field_validator

from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

from collections import Counter

warnings.filterwarnings("ignore")


C:\Users\jeeey\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Provider and model routing

Two switches control all LLM nodes (generator, agents, judge) at once:

```python
CONFIG["provider"]           = "groq"   # or "gemini"
CONFIG["execution_profile"]  = "dev"    # or "final"
```

**Groq** (recommended — much higher quota):

| Profile | Model | RPM | RPD | TPM |
|---|---|---|---|---|
| dev | `llama-3.1-8b-instant` | 30 | 14.4K | 6K |
| final | `meta-llama/llama-4-scout-17b-16e-instruct` | 30 | 1K | **30K** |

Use `llama-4-scout` for the final run — its 30K TPM is the key advantage for long RAG contexts.

**Gemini** (fallback):

| Profile | Model |
|---|---|
| dev | `gemini-2.0-flash-lite` |
| final | `gemini-2.0-flash` |

`max_rpm` is set automatically when the config cell runs (25 for Groq, 10 for Gemini). Override manually if needed.


## 1. Enter API key

In [3]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = input("Enter your GROQ_API_KEY (leave blank if using Gemini): ").strip()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = input("Enter your GOOGLE_API_KEY (leave blank if using Groq): ").strip()

GROQ_API_KEY   = os.environ.get("GROQ_API_KEY", "")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")


## 2. Configuration

In [4]:
CONFIG: Dict[str, Any] = {
    "random_seed": 42,

    # dataset size
    "pilot_n_questions": 10,
    "full_n_questions": 30,
    "use_pilot": False,

    # retrieval / agent loop controls
    "max_context_retries": 1,   # how many times context evaluator can reject before forcing through
    "max_repair_retrievals": 1,  # how many times repair agent can request fresh retrieval
    "chunk_words": 180,
    "chunk_overlap_words": 30,
    "bm25_top_k": 8,
    "dense_top_k": 8,
    "rerank_top_k": 4,

    # local retrieval models
    "dense_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "reranker_model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",

    # -------------------------------------------------------------------------
    # Provider switch
    # Set to "groq" or "gemini" — this controls which LLM backend is used
    # for ALL nodes (generator, agents, judge) in a single place.
    # -------------------------------------------------------------------------
    "provider": "groq",

    # model routing
    # "dev"   -> fast / cheap model — use for debugging and pilot runs
    # "final" -> stronger model — use only for the final reported run
    # Change only execution_profile to switch tier; change provider above to switch backend.
    "execution_profile": "final",

    # Groq models
    # dev:   llama-3.1-8b-instant  — 30 RPM, 14.4K RPD, 6K TPM  (safe for heavy iteration)
    # final: llama-4-scout-17b     — 30 RPM, 1K RPD,   30K TPM  (best TPM for long RAG contexts)
    "groq_dev_generator_model": "llama-3.1-8b-instant",
    "groq_dev_agent_model":     "llama-3.1-8b-instant",
    "groq_dev_judge_model":     "llama-3.1-8b-instant",
    "groq_final_generator_model": "meta-llama/llama-4-scout-17b-16e-instruct",
    "groq_final_agent_model":     "meta-llama/llama-4-scout-17b-16e-instruct",
    "groq_final_judge_model":     "meta-llama/llama-4-scout-17b-16e-instruct",

    # Gemini models
    # dev:   gemini-2.0-flash-lite — lighter, higher quota
    # final: gemini-2.0-flash      — stronger model
    "gemini_dev_generator_model": "gemini-2.0-flash-lite",
    "gemini_dev_agent_model":     "gemini-2.0-flash-lite",
    "gemini_dev_judge_model":     "gemini-2.0-flash-lite",
    "gemini_final_generator_model": "gemini-2.0-flash",
    "gemini_final_agent_model":     "gemini-2.0-flash",
    "gemini_final_judge_model":     "gemini-2.0-flash",

    # temperatures
    "temperature_rewriter": 0.2,
    "temperature_context_eval": 0.0,
    "temperature_generator": 0.2,
    "temperature_critic": 0.0,
    "temperature_repair": 0.1,
    "temperature_judge": 0.0,

    # rate limiting: global RPM cap enforced across every LLM call in the notebook.
    # Groq free tier: 30 RPM for most models — use 25 as a safe buffer.
    # Gemini free tier: 10-15 RPM depending on model — use 10 to be safe.
    # This value is set automatically by resolve_model_name() below; you can override manually.
    "max_rpm": 25,

    # small sleep between questions as a secondary courtesy buffer —
    # the RateLimiter in Cell 23 handles the real throttling
    "inter_question_sleep_sec": 0.5,

    # retry settings for individual LLM calls
    "llm_max_retries": 3,
    "llm_retry_base_delay_sec": 5,

    # LLM judge: number of questions to judge per pipeline (set 0 to skip)
    # runs after the main experiment as a lightweight optional step
    "judge_sample_n": 5,
}


def resolve_model_name(role: str) -> str:
    provider = CONFIG["provider"]
    profile  = CONFIG["execution_profile"]
    if provider not in ("groq", "gemini"):
        raise ValueError(f"Unknown provider: {provider!r}. Must be 'groq' or 'gemini'.")
    if profile not in ("dev", "final"):
        raise ValueError(f"Unknown execution_profile: {profile!r}. Must be 'dev' or 'final'.")
    return CONFIG[f"{provider}_{profile}_{role}_model"]


# Auto-set a sensible max_rpm based on the active provider.
# You can always override this manually after the cell runs.
CONFIG["max_rpm"] = 25 if CONFIG["provider"] == "groq" else 10

print("Provider:         ", CONFIG["provider"])
print("Execution profile:", CONFIG["execution_profile"])
print("Generator model:  ", resolve_model_name("generator"))
print("Agent model:      ", resolve_model_name("agent"))
print("Judge model:      ", resolve_model_name("judge"))
print("Max RPM:          ", CONFIG["max_rpm"])

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])


Provider:          groq
Execution profile: final
Generator model:   meta-llama/llama-4-scout-17b-16e-instruct
Agent model:       meta-llama/llama-4-scout-17b-16e-instruct
Judge model:       meta-llama/llama-4-scout-17b-16e-instruct
Max RPM:           25


## 3. Dataset loading (FinanceBench)

In [5]:
def load_financebench_subset(n_questions: int = 30) -> pd.DataFrame:
    # Load the open-source FinanceBench subset (150 questions) from HuggingFace.
    # Each row has a question, gold answer, evidence list, and filing metadata.
    # We flatten the first evidence entry's fields for convenience since most
    # questions have a single piece of gold evidence.
    print("Loading FinanceBench from HuggingFace (PatronusAI/financebench)...")
    ds = load_dataset("PatronusAI/financebench", split="train")

    rows = []
    for item in ds:
        evidence_list = item.get("evidence", [])

        # Pull out the first evidence entry's fields — most questions have one
        first_ev        = evidence_list[0] if evidence_list else {}
        gold_ev_text    = first_ev.get("evidence_text", "")
        gold_ev_page    = first_ev.get("evidence_text_full_page", gold_ev_text)
        gold_ev_page_num= first_ev.get("evidence_page_num", -1)

        # Collect ALL full pages across all evidence entries — this is our corpus
        all_pages = []
        seen = set()
        for ev in evidence_list:
            page_text = ev.get("evidence_text_full_page", "")
            if page_text and page_text not in seen:
                all_pages.append({
                    "text": page_text,
                    "page_num": ev.get("evidence_page_num", -1),
                })
                seen.add(page_text)

        rows.append({
            "id":                item.get("financebench_id", ""),
            "question":          item.get("question", ""),
            "reference_answer":  item.get("answer", ""),
            "company":           item.get("company", ""),
            "doc_name":          item.get("doc_name", ""),
            "question_type":     item.get("question_type", "unknown"),
            "question_reasoning":item.get("question_reasoning", "unknown"),
            "gold_evidence_text":gold_ev_text,        # exact evidence string — used for retrieval hit
            "gold_evidence_page":gold_ev_page,        # full page containing evidence — used as corpus
            "gold_evidence_page_num": gold_ev_page_num,
            "all_evidence_pages":all_pages,            # all pages across evidence entries
            "expected_decision": "answer",
        })

    df = pd.DataFrame(rows)
    df = df.sample(
        n=min(n_questions, len(df)),
        random_state=CONFIG["random_seed"]
    ).reset_index(drop=True)

    print(f"Loaded {len(df)} questions covering {df['doc_name'].nunique()} unique filings.")
    return df


N_QUESTIONS = CONFIG["pilot_n_questions"] if CONFIG["use_pilot"] else CONFIG["full_n_questions"]
eval_df = load_financebench_subset(N_QUESTIONS)
eval_df[["id", "company", "doc_name", "question_type", "question", "reference_answer"]].head()


Loading FinanceBench from HuggingFace (PatronusAI/financebench)...


Loaded 30 questions covering 26 unique filings.


,id,company,doc_name,question_type,question,reference_answer
0,financebench_id_00005,Corning,CORNING_2022_10K,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...
1,financebench_id_06655,Amazon,AMAZON_2017_10K,metrics-generated,What is Amazon's FY2017 days payable outstandi...,93.86
2,financebench_id_00080,Paypal,PAYPAL_2022_10K,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...
3,financebench_id_01244,CVS Health,CVSHEALTH_2022_10K,domain-relevant,Has CVS Health paid dividends to common shareh...,"Yes, CVS paid a $ 0.55 dividend per share ever..."
4,financebench_id_00790,CVS Health,CVSHEALTH_2022_10K,domain-relevant,Is CVS Health a capital-intensive business bas...,"Yes, CVS Health requires an extensive asset ba..."


## 4. Filing text loading

In [6]:
# Cache indexed by doc_name so questions sharing a filing don't rebuild the corpus
_filing_pages_cache: Dict[str, List[Dict[str, Any]]] = {}


def clean_text(text: str) -> str:
    # Collapse extra whitespace — financial pages have lots of it from PDF extraction
    text = re.sub(r" {2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def get_filing_pages_for_question(row: pd.Series) -> List[Dict[str, Any]]:
    # Returns all evidence pages for a question's filing, cached by doc_name.
    # Using the pre-extracted full-page text from FinanceBench avoids PDF parsing.
    # The corpus for each question = all unique evidence pages belonging to its filing.
    doc_name = row["doc_name"]

    if doc_name in _filing_pages_cache:
        return _filing_pages_cache[doc_name]

    pages = []
    for entry in row["all_evidence_pages"]:
        raw_text = entry.get("text", "")
        if raw_text:
            pages.append({
                "doc_name":  doc_name,
                "company":   row["company"],
                "page_num":  entry.get("page_num", -1),
                "text":      clean_text(raw_text),
            })

    _filing_pages_cache[doc_name] = pages
    return pages


## 5. Chunking for contextual RAG

In [7]:
def chunk_text_by_words(text: str, chunk_words: int, overlap_words: int) -> List[str]:
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0
    step = max(1, chunk_words - overlap_words)

    while start < len(words):
        end = min(len(words), start + chunk_words)
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end == len(words):
            break
        start += step

    return chunks


def build_contextual_chunks(page: Dict[str, Any]) -> List[Dict[str, Any]]:
    # For SEC filings, each page is a meaningful unit (balance sheet, MD&A paragraph, etc.).
    # We prepend filing metadata to every chunk so the model always knows which company,
    # filing, and page a passage came from — critical for financial multi-source QA.
    doc_name = page["doc_name"]
    company  = page["company"]
    page_num = page["page_num"]
    text     = page.get("text", "")

    raw_chunks = chunk_text_by_words(text, CONFIG["chunk_words"], CONFIG["chunk_overlap_words"])

    chunks = []
    for idx, chunk in enumerate(raw_chunks):
        contextual_text = (
            f"Company: {company}\n"
            f"Filing: {doc_name}\n"
            f"Page: {page_num}\n"
            f"Chunk ID: {idx}\n"
            f"Content: {chunk}"
        )
        chunks.append({
            "doc_name":        doc_name,
            "company":         company,
            "page_num":        page_num,
            "chunk_id":        idx,
            "raw_chunk":       chunk,
            "contextual_chunk":contextual_text,
        })
    return chunks


## 6. Retriever components

In [8]:
print("Loading dense embedder...")
dense_model = SentenceTransformer(CONFIG["dense_model_name"])
print("Loading reranker...")
reranker = CrossEncoder(CONFIG["reranker_model_name"])


@dataclass
class RetrievedChunk:
    doc_name: str
    company: str
    page_num: int
    chunk_id: int
    raw_chunk: str
    contextual_chunk: str
    score: float
    source: str


def _rrf_score(rank: int, k: int = 60) -> float:
    # Reciprocal Rank Fusion formula: 1 / (k + rank)
    # k=60 is the standard constant from the original RRF paper.
    # Higher rank (closer to 0) gives a bigger contribution.
    return 1.0 / (k + rank + 1)  # +1 because rank is 0-indexed


class CorpusIndex:
    def __init__(self, chunks: List[Dict[str, Any]]):
        self.df = pd.DataFrame(chunks).copy()
        self.df["bm25_tokens"] = self.df["contextual_chunk"].str.lower().str.split()
        self.bm25 = BM25Okapi(self.df["bm25_tokens"].tolist())
        self.embeddings = dense_model.encode(
            self.df["contextual_chunk"].tolist(),
            show_progress_bar=False,
            normalize_embeddings=True,
        )

    def bm25_search(self, query: str, top_k: int) -> List[RetrievedChunk]:
        tokenized = query.lower().split()
        scores = self.bm25.get_scores(tokenized)
        top_idx = np.argsort(scores)[::-1][:top_k]
        out = []
        for idx in top_idx:
            row = self.df.iloc[idx]
            out.append(RetrievedChunk(
                doc_name=row["doc_name"],
                company=row["company"],
                page_num=int(row["page_num"]),
                chunk_id=int(row["chunk_id"]),
                raw_chunk=row["raw_chunk"],
                contextual_chunk=row["contextual_chunk"],
                score=float(scores[idx]),
                source="bm25",
            ))
        return out

    def dense_search(self, query: str, top_k: int) -> List[RetrievedChunk]:
        q_emb = dense_model.encode([query], show_progress_bar=False, normalize_embeddings=True)
        sims = cosine_similarity(q_emb, self.embeddings)[0]
        top_idx = np.argsort(sims)[::-1][:top_k]
        out = []
        for idx in top_idx:
            row = self.df.iloc[idx]
            out.append(RetrievedChunk(
                doc_name=row["doc_name"],
                company=row["company"],
                page_num=int(row["page_num"]),
                chunk_id=int(row["chunk_id"]),
                raw_chunk=row["raw_chunk"],
                contextual_chunk=row["contextual_chunk"],
                score=float(sims[idx]),
                source="dense",
            ))
        return out

    def hybrid_search(
        self,
        query: str,
        bm25_top_k: int,
        dense_top_k: int,
        rerank_top_k: int,
    ) -> List[RetrievedChunk]:
        bm25_results = self.bm25_search(query, bm25_top_k)
        dense_results = self.dense_search(query, dense_top_k)

        # Use Reciprocal Rank Fusion to merge the two ranked lists.
        # The key insight: BM25 and dense scores live on completely different
        # numerical scales, so directly comparing them is meaningless.
        # RRF only uses each result's *rank position* — not its raw score —
        # which makes it scale-agnostic and a much fairer fusion method.
        rrf_scores: Dict[Tuple[str, int], float] = {}
        chunk_map: Dict[Tuple[str, int], RetrievedChunk] = {}

        for rank, item in enumerate(bm25_results):
            key = (item.doc_name, item.page_num, item.chunk_id)
            rrf_scores[key] = rrf_scores.get(key, 0.0) + _rrf_score(rank)
            chunk_map[key] = item

        for rank, item in enumerate(dense_results):
            key = (item.doc_name, item.page_num, item.chunk_id)
            rrf_scores[key] = rrf_scores.get(key, 0.0) + _rrf_score(rank)
            if key not in chunk_map:
                chunk_map[key] = item

        # Sort by combined RRF score, then rerank the top candidates
        sorted_keys = sorted(rrf_scores.keys(), key=lambda k: rrf_scores[k], reverse=True)
        candidates = [chunk_map[k] for k in sorted_keys]

        if not candidates:
            return []

        pairs = [(query, c.contextual_chunk) for c in candidates]
        rerank_scores = reranker.predict(pairs, show_progress_bar=False)
        for c, s in zip(candidates, rerank_scores):
            c.score = float(s)
            c.source = "hybrid_reranked"

        candidates = sorted(candidates, key=lambda x: x.score, reverse=True)
        return candidates[:rerank_top_k]


Loading dense embedder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 7. Gemini LLM clients

In [9]:
def make_llm(model_name: str, temperature: float):
    # Returns the appropriate LangChain chat model based on CONFIG["provider"].
    # Both ChatGroq and ChatGoogleGenerativeAI support .with_structured_output()
    # so everything downstream works identically regardless of which is active.
    if CONFIG["provider"] == "groq":
        return ChatGroq(
            model=model_name,
            temperature=temperature,
            groq_api_key=GROQ_API_KEY,
        )
    elif CONFIG["provider"] == "gemini":
        return ChatGoogleGenerativeAI(
            model=model_name,
            temperature=temperature,
            google_api_key=GOOGLE_API_KEY,
        )
    else:
        raise ValueError(f"Unknown provider: {CONFIG['provider']!r}. Must be 'groq' or 'gemini'.")


rewriter_llm      = make_llm(resolve_model_name("agent"),     CONFIG["temperature_rewriter"])
context_eval_llm  = make_llm(resolve_model_name("agent"),     CONFIG["temperature_context_eval"])
generator_llm     = make_llm(resolve_model_name("generator"), CONFIG["temperature_generator"])
critic_llm        = make_llm(resolve_model_name("agent"),     CONFIG["temperature_critic"])
repair_llm        = make_llm(resolve_model_name("agent"),     CONFIG["temperature_repair"])
judge_llm         = make_llm(resolve_model_name("judge"),     CONFIG["temperature_judge"])


## 8. Structured outputs for agents

In [10]:
# Pydantic models define the schema each agent must return.
# We use Gemini's native structured output (via .with_structured_output) rather
# than PydanticOutputParser, which worked by injecting JSON format instructions
# into the prompt and hoping the model followed them — unreliable with Gemini.
# Native structured output uses function-calling under the hood, which enforces
# the schema at the API level and is far more robust.

class QueryRewriteOutput(BaseModel):
    rewritten_query: str = Field(description="A concise rewritten query optimized for retrieval")


class ContextEvalOutput(BaseModel):
    relevant: bool = Field(description="Whether the retrieved context is relevant and sufficient")
    reason: str = Field(description="Short reason")

    # llama-4-scout sometimes returns "true"/"false" as strings instead of
    # proper JSON booleans. This validator catches that before Pydantic rejects it.
    @field_validator('relevant', mode='before')
    @classmethod
    def coerce_bool(cls, v):
        if isinstance(v, str):
            return v.strip().lower() == 'true'
        return v


class CriticOutput(BaseModel):
    decision: Literal["accept", "repair"] = Field(
        description="Whether to accept the answer or route to the repair agent"
    )
    reason: str = Field(description="Short critique")


class RepairOutput(BaseModel):
    decision: Literal["accept", "warn", "refuse"] = Field(
        description=(
            "Final decision after repair. Use 'accept' when you have a revised answer — "
            "even a partial or approximate one. Use 'warn' only when the context is genuinely ambiguous. "
            "Use 'refuse' ONLY if the question is entirely out of scope or the wrong filing — "
            "never refuse just because the question is complex or requires a calculation."
        )
    )
    revised_answer: str = Field(
        description="The final revised answer. Required for 'accept'. For 'warn', include what you can state with confidence."
    )
    needs_new_retrieval: bool = Field(
        description="True only if the existing context is genuinely insufficient and new retrieval would help"
    )
    reason: str = Field(description="Short rationale")


class JudgeOutput(BaseModel):
    score: float = Field(description="Overall correctness score between 0 and 1")
    claims_covered: float = Field(
        description=(
            "Fraction of key factual claims in the reference answer that are present "
            "in the candidate answer. E.g. if the reference states 3 facts and the "
            "candidate covers 2, this is 0.67. For financial answers treat each "
            "distinct number, ratio, or named metric as a separate claim."
        )
    )
    reason: str = Field(description="Short explanation")


## 9. Prompt templates

In [11]:
# Prompts no longer contain {format_instructions} — that was only needed for
# PydanticOutputParser. With native structured output the schema is enforced
# at the API level, so the prompt just needs to describe the task clearly.

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You rewrite financial questions into concise search-optimized queries. "
        "Preserve company names, fiscal periods (FY2022, Q3 2023), and financial metric names exactly.",
    ),
    ("human", "Original question:\n{question}"),
])

context_eval_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You judge whether retrieved context is relevant and sufficient to answer a financial question. "
        "Mark as not relevant only if the context is clearly from the wrong company/period, or completely empty. "
        "Partial tables or incomplete passages still count as relevant if they contain the right metric.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}"),
])

generator_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a financial analyst answering questions using only the retrieved filing context. "
        "Be precise with numbers — include units (millions, billions, %), fiscal periods, and line item names. "
        "If the context does not contain the answer, say so explicitly rather than estimating.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}"),
])

critic_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a critic for a financial RAG system. Decide only whether the current answer should be "
        "accepted or sent to the repair agent. Only request repair if the answer contradicts the context, "
        "uses the wrong fiscal period, reports the wrong metric, or fabricates numbers not in the context. "
        "An incomplete but numerically honest answer should be accepted.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}\n\nCurrent answer:\n{answer}"),
])

repair_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You repair a financial RAG answer after critique. Your default action is to ACCEPT with a revised answer. "
        "Use the retrieved context to correct errors in the original answer. "
        "Pay close attention to: correct fiscal year/quarter, correct units (millions vs billions), "
        "correct sign (positive vs negative), and the right line item name. "
        "Set decision='accept' whenever you can produce any useful revised answer — even partial. "
        "Set decision='warn' only if the context is genuinely ambiguous. "
        "Set decision='refuse' ONLY if the question is entirely off-topic for this filing. "
        "Set needs_new_retrieval=true only if critical data is completely absent from the context. "
        "Return the final user-facing answer in revised_answer."
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}\n\n"
        "Original answer:\n{answer}\n\nCritic feedback:\n{critique}",
    ),
])

judge_correctness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. "
        "1 = correct value, correct units, correct period. "
        "0 = wrong number, wrong company, wrong period, or completely missing. "
        "Give partial credit for answers that are close but have unit errors (e.g. millions vs billions). "
        "Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate.",
    ),
    (
        "human",
        "Question:\n{question}\n\nReference answer:\n{reference_answer}\n\n"
        "Candidate answer:\n{candidate_answer}",
    ),
])

judge_faithfulness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. "
        "1 = every number and claim is directly supported by the context. "
        "0 = answer contains numbers or claims not present in the context (hallucinated). "
        "For financial answers, each number, ratio, and named metric counts as a separate claim. "
        "Also set claims_covered: fraction of claims in the candidate that are supported by the context.",
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}\n\n"
        "Candidate answer:\n{candidate_answer}",
    ),
])


## 10. Helper functions for agent calls

In [12]:
class RateLimiter:
    # Sliding-window rate limiter.
    # Keeps a queue of the last N call timestamps. Before every LLM call,
    # wait() checks if we've already made max_rpm calls in the past 60s.
    # If so, it sleeps exactly long enough for the oldest call to leave
    # the window — no over-sleeping, no quota errors.
    def __init__(self, max_rpm: int):
        self.max_rpm    = max_rpm
        self.window     = 60.0          # one-minute rolling window
        self.timestamps: deque = deque()
        self._lock      = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()

            # Remove timestamps older than the 60-second window
            while self.timestamps and now - self.timestamps[0] >= self.window:
                self.timestamps.popleft()

            if len(self.timestamps) >= self.max_rpm:
                # Sleep until the oldest timestamp is 60s old and drops out
                sleep_for = self.window - (now - self.timestamps[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] {self.max_rpm} RPM cap reached — waiting {sleep_for:.1f}s")
                    time.sleep(sleep_for)

            # Record this call
            self.timestamps.append(time.time())


# One global limiter — every LLM call in the notebook goes through this,
# whether it's a graph node, a generator call, or the judge
_rate_limiter = RateLimiter(max_rpm=CONFIG["max_rpm"])


def safe_invoke_structured(
    llm,
    schema_class,
    prompt,
    variables: Dict[str, Any],
    fallback: BaseModel,
) -> BaseModel:
    # Invoke an LLM with native structured output (schema enforced at API level).
    # Blocks via the global rate limiter before each attempt, then retries with
    # exponential backoff on failure. Always prints a warning if it falls back
    # so failures are never silent.
    max_retries = CONFIG["llm_max_retries"]
    base_delay  = CONFIG["llm_retry_base_delay_sec"]

    for attempt in range(max_retries):
        try:
            _rate_limiter.wait()   # enforce the global RPM cap before firing
            structured_llm = llm.with_structured_output(schema_class)
            chain = prompt | structured_llm
            return chain.invoke(variables)
        except Exception as e:
            is_last = (attempt == max_retries - 1)
            delay   = base_delay * (2 ** attempt)
            print(
                f"  [WARN] {schema_class.__name__} attempt {attempt + 1}/{max_retries} "
                f"failed: {type(e).__name__}: {e}"
            )
            if is_last:
                print(f"  [WARN] All retries exhausted for {schema_class.__name__}, using fallback.")
                return fallback
            time.sleep(delay)

    return fallback


def safe_invoke_llm(chain, variables: Dict[str, Any], fallback_text: str = "") -> str:
    # Invoke a plain (non-structured) LLM chain with rate limiting, retry and backoff.
    # Returns the .content string, or fallback_text if all retries fail.
    max_retries = CONFIG["llm_max_retries"]
    base_delay  = CONFIG["llm_retry_base_delay_sec"]

    for attempt in range(max_retries):
        try:
            _rate_limiter.wait()   # enforce the global RPM cap before firing
            return chain.invoke(variables).content
        except Exception as e:
            is_last = (attempt == max_retries - 1)
            delay   = base_delay * (2 ** attempt)
            print(f"  [WARN] LLM call attempt {attempt + 1}/{max_retries} failed: {type(e).__name__}: {e}")
            if is_last:
                print(f"  [WARN] All retries exhausted, returning fallback text.")
                return fallback_text
            time.sleep(delay)

    return fallback_text


def format_retrieved_context(chunks: List[RetrievedChunk]) -> str:
    parts = []
    for i, c in enumerate(chunks, start=1):
        parts.append(
            f"[Doc {i}] Company: {c.company} | Filing: {c.doc_name} | Page: {c.page_num}\n"
            f"Content: {c.raw_chunk}"
        )
    return "\n\n".join(parts)


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    return list(dict.fromkeys([c.doc_name for c in chunks]))


## 11. Build per-question corpus index

For each question we build a small index from the pre-extracted filing pages in FinanceBench.
The corpus = all unique evidence pages for this question's filing (cached by doc_name).
This gives a real (if small) retrieval challenge without requiring full PDF parsing.


In [13]:
def build_question_corpus_index(row: pd.Series) -> CorpusIndex:
    # Get the filing pages for this question (cached after first call per doc_name)
    pages = get_filing_pages_for_question(row)

    if not pages:
        raise ValueError(f"No filing pages found for doc_name='{row['doc_name']}'")

    # Chunk each page and collect all chunks into one flat list
    chunks: List[Dict[str, Any]] = []
    for page in pages:
        chunks.extend(build_contextual_chunks(page))

    return CorpusIndex(chunks)


## 12. Pipeline implementations

In [14]:
def run_simple_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    retrieved = index.dense_search(question, top_k=CONFIG["rerank_top_k"])
    context = format_retrieved_context(retrieved)

    raw_answer = safe_invoke_llm(
        generator_prompt | generator_llm,
        {"question": question, "context": context},
        fallback_text="",
    )
    answer = normalize_answer_text(raw_answer)
    latency = time.time() - start

    return {
        "pipeline": "simple_rag",
        "rewritten_query": question,
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
        "context_retries": 0,
        "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None,
        "repair_used": False,
        "repair_decision": None,
        "final_answer": answer,
        "latency_sec": latency,
    }


def run_advanced_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    retrieved = index.hybrid_search(
        question,
        bm25_top_k=CONFIG["bm25_top_k"],
        dense_top_k=CONFIG["dense_top_k"],
        rerank_top_k=CONFIG["rerank_top_k"],
    )
    context = format_retrieved_context(retrieved)

    raw_answer = safe_invoke_llm(
        generator_prompt | generator_llm,
        {"question": question, "context": context},
        fallback_text="",
    )
    answer = normalize_answer_text(raw_answer)
    latency = time.time() - start

    return {
        "pipeline": "advanced_rag",
        "rewritten_query": question,
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
        "context_retries": 0,
        "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None,
        "repair_used": False,
        "repair_decision": None,
        "final_answer": answer,
        "latency_sec": latency,
    }


## 13. LangGraph state and nodes

In [15]:
class GraphState(TypedDict, total=False):
    question: str
    doc_name: str                    # FinanceBench filing identifier
    gold_evidence_text: str          # gold evidence string for retrieval evaluation
    index: CorpusIndex
    rewritten_query: str
    retrieved_chunks: List[RetrievedChunk]
    retrieved_doc_names: List[str]
    context_retries: int
    context_eval_relevant: bool
    generated_answer: str
    critic_decision: str
    critic_reason: str
    repair_used: bool
    repair_decision: str
    repair_reason: str
    needs_new_retrieval: bool
    repair_retrieval_count: int
    is_repair_retrieval: bool
    final_answer: str


def node_query_rewriter(state: GraphState) -> GraphState:
    result = safe_invoke_structured(
        rewriter_llm,
        QueryRewriteOutput,
        rewrite_prompt,
        {"question": state["question"]},
        QueryRewriteOutput(rewritten_query=state["question"]),
    )
    return {"rewritten_query": normalize_answer_text(result.rewritten_query)}


def node_hybrid_retriever(state: GraphState) -> GraphState:
    query = state.get("rewritten_query", state["question"])
    retrieved = state["index"].hybrid_search(
        query,
        bm25_top_k=CONFIG["bm25_top_k"],
        dense_top_k=CONFIG["dense_top_k"],
        rerank_top_k=CONFIG["rerank_top_k"],
    )
    return {
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
    }


def node_context_evaluator(state: GraphState) -> GraphState:
    context = format_retrieved_context(state.get("retrieved_chunks", []))
    result = safe_invoke_structured(
        context_eval_llm,
        ContextEvalOutput,
        context_eval_prompt,
        {"question": state["question"], "context": context},
        ContextEvalOutput(relevant=True, reason="Fallback accept"),
    )
    return {
        "context_eval_relevant": result.relevant,
        "context_retries": state.get("context_retries", 0),
    }


def route_context(state: GraphState) -> str:
    relevant = state.get("context_eval_relevant", True)
    retries = state.get("context_retries", 0)
    if relevant:
        return "generator"
    if retries < CONFIG["max_context_retries"]:
        return "retry_retrieval"
    # Hit the retry cap — force through to generator anyway
    return "generator"


def node_increment_context_retry(state: GraphState) -> GraphState:
    return {"context_retries": state.get("context_retries", 0) + 1}


def node_generator(state: GraphState) -> GraphState:
    context = format_retrieved_context(state.get("retrieved_chunks", []))
    raw_answer = safe_invoke_llm(
        generator_prompt | generator_llm,
        {"question": state["question"], "context": context},
        fallback_text="",
    )
    answer = normalize_answer_text(raw_answer)
    return {"generated_answer": answer, "final_answer": answer}


def node_critic(state: GraphState) -> GraphState:
    context = format_retrieved_context(state.get("retrieved_chunks", []))
    result = safe_invoke_structured(
        critic_llm,
        CriticOutput,
        critic_prompt,
        {
            "question": state["question"],
            "context": context,
            "answer": state.get("generated_answer", ""),
        },
        CriticOutput(decision="accept", reason="Fallback accept"),
    )
    return {
        "critic_decision": result.decision,
        "critic_reason": result.reason,
    }


def route_critic(state: GraphState) -> str:
    # If this generation pass was triggered by the repair agent requesting new retrieval,
    # skip re-criticism to avoid creating an infinite repair loop
    if state.get("is_repair_retrieval", False):
        return "end"
    return "repair" if state.get("critic_decision") == "repair" else "end"


def node_repair(state: GraphState) -> GraphState:
    context = format_retrieved_context(state.get("retrieved_chunks", []))
    result = safe_invoke_structured(
        repair_llm,
        RepairOutput,
        repair_prompt,
        {
            "question": state["question"],
            "context": context,
            "answer": state.get("generated_answer", ""),
            "critique": state.get("critic_reason", "Needs repair"),
        },
        RepairOutput(
            decision="accept",
            revised_answer=state.get("generated_answer", ""),
            needs_new_retrieval=False,
            reason="Fallback repair",
        ),
    )

    # Track how many times repair has requested a fresh retrieval
    repair_retrieval_count = state.get("repair_retrieval_count", 0)
    if result.needs_new_retrieval:
        repair_retrieval_count += 1

    return {
        "repair_used": True,
        "repair_decision": result.decision,
        "repair_reason": result.reason,
        "needs_new_retrieval": result.needs_new_retrieval,
        "repair_retrieval_count": repair_retrieval_count,
        "final_answer": normalize_answer_text(result.revised_answer),
    }


def route_repair(state: GraphState) -> str:
    # Send back for fresh retrieval only if the repair agent asked for it
    # AND we haven't hit the configured cap (to prevent runaway loops)
    needs_new = state.get("needs_new_retrieval", False)
    count = state.get("repair_retrieval_count", 0)
    if needs_new and count <= CONFIG["max_repair_retrievals"]:
        return "re_retrieve"
    return "end"


def node_mark_repair_retrieval(state: GraphState) -> GraphState:
    # Set a flag so the critic knows to skip re-criticism after this retrieval pass
    return {"is_repair_retrieval": True}


def build_agentic_graph():
    graph = StateGraph(GraphState)

    graph.add_node("query_rewriter",          node_query_rewriter)
    graph.add_node("hybrid_retriever",         node_hybrid_retriever)
    graph.add_node("context_evaluator",        node_context_evaluator)
    graph.add_node("increment_context_retry",  node_increment_context_retry)
    graph.add_node("generator",                node_generator)
    graph.add_node("critic",                   node_critic)
    graph.add_node("repair",                   node_repair)
    graph.add_node("mark_repair_retrieval",    node_mark_repair_retrieval)

    graph.set_entry_point("query_rewriter")
    graph.add_edge("query_rewriter",         "hybrid_retriever")
    graph.add_edge("hybrid_retriever",        "context_evaluator")

    graph.add_conditional_edges(
        "context_evaluator",
        route_context,
        {"generator": "generator", "retry_retrieval": "increment_context_retry"},
    )
    graph.add_edge("increment_context_retry", "hybrid_retriever")
    graph.add_edge("generator",               "critic")

    graph.add_conditional_edges(
        "critic",
        route_critic,
        {"repair": "repair", "end": END},
    )

    # Repair agent either ends or loops back for fresh retrieval (capped by max_repair_retrievals)
    graph.add_conditional_edges(
        "repair",
        route_repair,
        {"re_retrieve": "mark_repair_retrieval", "end": END},
    )
    # After marking the repair-retrieval flag, re-run retrieval then generation
    graph.add_edge("mark_repair_retrieval",   "hybrid_retriever")

    return graph.compile()


agentic_graph = build_agentic_graph()


def run_agentic_rag(question: str, doc_name: str, gold_evidence_text: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    state: GraphState = {
        "question": question,
        "doc_name": doc_name,
        "gold_evidence_text": gold_evidence_text,
        "index": index,
        "context_retries": 0,
        "repair_used": False,
        "repair_retrieval_count": 0,
        "is_repair_retrieval": False,
    }
    out = agentic_graph.invoke(state)
    latency = time.time() - start

    generated_answer = normalize_answer_text(out.get("generated_answer", ""))
    final_answer      = normalize_answer_text(out.get("final_answer", generated_answer))
    rewritten_query   = normalize_answer_text(out.get("rewritten_query", question))

    return {
        "pipeline": "agentic_rag_langgraph",
        "rewritten_query": rewritten_query,
        "retrieved_chunks": out.get("retrieved_chunks", []),
        "retrieved_doc_names": out.get("retrieved_doc_names", []),
        "context_retries": out.get("context_retries", 0),
        "context_eval_relevant": out.get("context_eval_relevant"),
        "generated_answer": generated_answer,
        "critic_decision": out.get("critic_decision"),
        "repair_used": out.get("repair_used", False),
        "repair_decision": out.get("repair_decision"),
        "repair_retrieval_count": out.get("repair_retrieval_count", 0),
        "final_answer": final_answer,
        "latency_sec": latency,
    }


## 14. Evaluation helpers

In [16]:
def llm_judge_correctness(
    question: str,
    reference_answer: str,
    candidate_answer: str,
) -> Tuple[float, float, str]:
    # Returns (score, claims_covered, reason)
    result = safe_invoke_structured(
        judge_llm,
        JudgeOutput,
        judge_correctness_prompt,
        {
            "question": question,
            "reference_answer": reference_answer,
            "candidate_answer": candidate_answer,
        },
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
    )
    score          = max(0.0, min(1.0, float(result.score)))
    claims_covered = max(0.0, min(1.0, float(result.claims_covered)))
    return score, claims_covered, result.reason


def llm_judge_faithfulness(
    question: str,
    context: str,
    candidate_answer: str,
) -> Tuple[float, float, str]:
    # Returns (score, claims_covered, reason)
    result = safe_invoke_structured(
        judge_llm,
        JudgeOutput,
        judge_faithfulness_prompt,
        {
            "question": question,
            "context": context,
            "candidate_answer": candidate_answer,
        },
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
    )
    score          = max(0.0, min(1.0, float(result.score)))
    claims_covered = max(0.0, min(1.0, float(result.claims_covered)))
    return score, claims_covered, result.reason


def _extract_numbers(text: str) -> List[float]:
    # Extract all numeric values from financial text, normalising common formats.
    # Handles: $1,234.5  /  1.2B  /  (45.6)  /  34%  /  1,234 million
    text = text.lower()

    # Replace written multipliers with numeric equivalents before parsing
    multipliers = [("trillion", 1e12), ("billion", 1e9), ("million", 1e6), ("thousand", 1e3)]
    for word, mult in multipliers:
        # e.g. "12.3 billion" → "12300000000.0"
        text = re.sub(
            rf"([\d,\.]+)\s*{word}",
            lambda m, mult=mult: str(float(m.group(1).replace(",", "")) * mult),
            text,
        )

    # Extract all remaining numbers (strip $ % commas, handle parentheses as negative)
    raw = re.findall(r"\(?[\$]?[\d,]+\.?\d*\)?", text)
    numbers = []
    for tok in raw:
        negative = tok.startswith("(") and tok.endswith(")")
        cleaned  = re.sub(r"[^\d\.]", "", tok)
        try:
            val = float(cleaned)
            numbers.append(-val if negative else val)
        except ValueError:
            pass
    return numbers


def numerical_accuracy_score(prediction: Any, reference: Any, tolerance: float = 0.01) -> float:
    # Check whether the prediction contains at least one number that matches
    # a number in the reference within `tolerance` (default 1%).
    # Returns 1.0 if any reference number is matched, 0.0 otherwise.
    # This is much more meaningful than token F1 for financial answers like "$1,577.00".
    pred_nums = _extract_numbers(normalize_answer_text(prediction))
    ref_nums  = _extract_numbers(normalize_answer_text(reference))

    if not ref_nums:
        return 1.0   # no numbers to check — not a numerical question, skip penalising

    if not pred_nums:
        return 0.0   # reference has numbers but prediction has none

    for ref_val in ref_nums:
        for pred_val in pred_nums:
            if ref_val == 0 and pred_val == 0:
                return 1.0
            if ref_val != 0 and abs(pred_val - ref_val) / abs(ref_val) <= tolerance:
                return 1.0

    return 0.0


def retrieval_evidence_hit(retrieved_chunks: List[RetrievedChunk], gold_evidence_text: str) -> float:
    # Strict retrieval hit: at least one retrieved chunk must have >50% token overlap
    # with the gold evidence string. This is much more meaningful than title overlap
    # since FinanceBench provides the exact sentence/paragraph that contains the answer.
    if not gold_evidence_text or not retrieved_chunks:
        return 0.0

    gold_tokens = set(normalize_for_eval(gold_evidence_text).split())
    if not gold_tokens:
        return 0.0

    for chunk in retrieved_chunks:
        chunk_tokens  = set(normalize_for_eval(chunk.raw_chunk).split())
        if not chunk_tokens:
            continue
        overlap = len(gold_tokens & chunk_tokens) / len(gold_tokens)
        if overlap >= 0.5:
            return 1.0

    return 0.0


def normalize_answer_text(answer: Any) -> str:
    if answer is None:
        return ""
    if isinstance(answer, str):
        return answer.strip()
    if isinstance(answer, list):
        return " ".join(str(x) for x in answer if x is not None).strip()
    if isinstance(answer, dict):
        for key in ["final_answer", "answer", "text", "content", "generated_answer"]:
            if key in answer:
                return normalize_answer_text(answer[key])
        return str(answer).strip()
    return str(answer).strip()


def normalize_for_eval(text: Any) -> str:
    text = normalize_answer_text(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match_score(prediction: Any, reference: Any) -> float:
    pred = normalize_for_eval(prediction)
    ref  = normalize_for_eval(reference)
    return 1.0 if pred == ref else 0.0


def token_f1_score(prediction: Any, reference: Any) -> float:
    pred_tokens = normalize_for_eval(prediction).split()
    ref_tokens  = normalize_for_eval(reference).split()

    if len(pred_tokens) == 0 and len(ref_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    common  = Counter(pred_tokens) & Counter(ref_tokens)
    n_same  = sum(common.values())
    if n_same == 0:
        return 0.0

    precision = n_same / len(pred_tokens)
    recall    = n_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_decision_from_text(answer: Any) -> str:
    lower = normalize_answer_text(answer).lower()
    if any(x in lower for x in [
        "cannot answer", "insufficient", "not enough context",
        "do not know", "don't know", "unclear from context", "insufficient evidence",
    ]):
        return "warn"
    if any(x in lower for x in ["refuse", "cannot comply", "will not help", "won't help"]):
        return "refuse"
    return "answer"


## 15. Experiment runner

In [17]:
def run_all_pipelines(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    sleep_sec = CONFIG["inter_question_sleep_sec"]

    for q_idx, (_, row) in enumerate(tqdm(eval_df.iterrows(), total=len(eval_df), desc="Running questions")):
        qid                = row["id"]
        question           = row["question"]
        reference_answer   = row["reference_answer"]
        company            = row["company"]
        doc_name           = row["doc_name"]
        question_type      = row["question_type"]
        gold_evidence_text = row["gold_evidence_text"]
        expected_decision  = row["expected_decision"]

        try:
            index = build_question_corpus_index(row)
        except Exception as e:
            print(f"  Skipping {qid}: corpus build error: {e}")
            continue

        pipeline_outputs = [
            run_simple_rag(question, index),
            run_advanced_rag(question, index),
            run_agentic_rag(question, doc_name, gold_evidence_text, index),
        ]

        for out in pipeline_outputs:
            final_answer     = normalize_answer_text(out.get("final_answer", ""))
            generated_answer = normalize_answer_text(out.get("generated_answer", ""))
            rewritten_query  = normalize_answer_text(out.get("rewritten_query", question))

            predicted_decision  = compute_decision_from_text(final_answer)
            decision_accuracy   = 1.0 if predicted_decision == expected_decision else 0.0
            exact_match         = exact_match_score(final_answer, reference_answer)
            token_f1            = token_f1_score(final_answer, reference_answer)
            numerical_acc       = numerical_accuracy_score(final_answer, reference_answer)
            retrieval_hit       = retrieval_evidence_hit(
                out.get("retrieved_chunks", []), gold_evidence_text
            )

            rows.append({
                "id":                     qid,
                "question":               question,
                "company":                company,
                "doc_name":               doc_name,
                "question_type":          question_type,
                "reference_answer":       reference_answer,
                "gold_evidence_text":     gold_evidence_text,
                "expected_decision":      expected_decision,
                "pipeline":              out["pipeline"],
                "rewritten_query":        rewritten_query,
                "retrieved_doc_names":    out.get("retrieved_doc_names", []),
                "context_retries":        out.get("context_retries", 0),
                "context_eval_relevant":  out.get("context_eval_relevant"),
                "generated_answer":       generated_answer,
                "critic_decision":        out.get("critic_decision"),
                "repair_used":            out.get("repair_used", False),
                "repair_decision":        out.get("repair_decision"),
                "repair_retrieval_count": out.get("repair_retrieval_count", 0),
                "final_answer":           final_answer,
                "latency_sec":            out.get("latency_sec"),
                "exact_match":            exact_match,
                "token_f1":               token_f1,
                "numerical_accuracy":     numerical_acc,
                "predicted_decision":     predicted_decision,
                "decision_accuracy":      decision_accuracy,
                "retrieval_hit":          retrieval_hit,
                # LLM judge columns — filled in the optional judging cell below
                "llm_correctness_score":   None,
                "llm_correctness_reason":  None,
                "llm_claims_covered":      None,
                "llm_faithfulness_score":  None,
                "llm_faithfulness_reason": None,
            })

        if q_idx < len(eval_df) - 1:
            time.sleep(sleep_sec)

    return pd.DataFrame(rows)


## 16. Run experiment

**Before running the full experiment, check off each item below:**

| # | Setting | Pilot value | Full-run value | Where |
|---|---|---|---|---|
| 1 | `use_pilot` | `True` | **`False`** | Cell 6 CONFIG |
| 2 | `execution_profile` | `"dev"` | **`"final"`** | Cell 6 CONFIG |
| 3 | `judge_sample_n` | `3` | **`5`** | Cell 6 CONFIG |
| 4 | Save outputs | commented out | **Uncomment** | Cell 47 |

**Quota reminder for `llama-4-scout` (final profile):**
- 1K RPD limit — a 30-question run costs ~480 LLM calls across all pipelines + judge
- Don't re-run the same day if the first full run completes successfully
- If you hit RPD limit mid-run, set `use_pilot=True` and resume from the checkpoint CSV

The main run uses cheap automatic metrics (exact_match, token_f1, retrieval hit, latency)
so it is light on API quota. LLM judging runs afterwards on a small sample per pipeline.


In [18]:
results_df = run_all_pipelines(eval_df)
print(f"Total rows: {len(results_df)} ({len(eval_df)} questions × 3 pipelines)")
results_df.head()


Running questions:   0%|          | 0/30 [00:00<?, ?it/s]

  [WARN] ContextEvalOutput attempt 1/3 failed: BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool ContextEvalOutput did not match schema: errors: [`/relevant`: expected boolean, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '[\n  {\n    "name": "ContextEvalOutput",\n    "parameters": {\n      "relevant": "true",\n      "reason": "The retrieved context provides the necessary information to calculate working capital, including current assets and current liabilities."\n    }\n  }\n]'}}
  [WARN] ContextEvalOutput attempt 2/3 failed: BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool ContextEvalOutput did not match schema: errors: [`/relevant`: expected boolean, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '[\n  {\n    "name": "ContextEvalOutput",\n    "parameters": {\n      

,id,question,company,doc_name,question_type,reference_answer,gold_evidence_text,expected_decision,pipeline,rewritten_query,...,token_f1,numerical_accuracy,predicted_decision,decision_accuracy,retrieval_hit,llm_correctness_score,llm_correctness_reason,llm_claims_covered,llm_faithfulness_score,llm_faithfulness_reason
0,financebench_id_00005,Does Corning have positive working capital bas...,Corning,CORNING_2022_10K,domain-relevant,Yes. Corning had a positive working capital am...,Consolidated Balance Sheets\nCorning Incorpora...,answer,simple_rag,Does Corning have positive working capital bas...,...,0.173913,1.0,answer,1.0,1.0,None,None,None,None,None
1,financebench_id_00005,Does Corning have positive working capital bas...,Corning,CORNING_2022_10K,domain-relevant,Yes. Corning had a positive working capital am...,Consolidated Balance Sheets\nCorning Incorpora...,answer,advanced_rag,Does Corning have positive working capital bas...,...,0.152047,1.0,answer,1.0,1.0,None,None,None,None,None
2,financebench_id_00005,Does Corning have positive working capital bas...,Corning,CORNING_2022_10K,domain-relevant,Yes. Corning had a positive working capital am...,Consolidated Balance Sheets\nCorning Incorpora...,answer,agentic_rag_langgraph,Corning working capital FY2022,...,0.171429,1.0,answer,1.0,1.0,None,None,None,None,None
3,financebench_id_06655,What is Amazon's FY2017 days payable outstandi...,Amazon,AMAZON_2017_10K,metrics-generated,93.86,"Table of Contents\nAMAZON.COM, INC.\nCONSOLIDA...",answer,simple_rag,What is Amazon's FY2017 days payable outstandi...,...,0.000000,0.0,answer,1.0,1.0,None,None,None,None,None
4,financebench_id_06655,What is Amazon's FY2017 days payable outstandi...,Amazon,AMAZON_2017_10K,metrics-generated,93.86,"Table of Contents\nAMAZON.COM, INC.\nCONSOLIDA...",answer,advanced_rag,What is Amazon's FY2017 days payable outstandi...,...,0.000000,0.0,answer,1.0,1.0,None,None,None,None,None


## 17. Optional: LLM judge (correctness + faithfulness)

In [19]:
# Runs LLM-as-judge scoring on a small sample per pipeline.
# judge_sample_n controls how many questions per pipeline are judged (set 0 to skip).
# Each judged row costs 2 API calls (correctness + faithfulness).
# At judge_sample_n=3 and 3 pipelines that's 18 calls total — very lightweight.

def run_llm_judging(
    results_df: pd.DataFrame,
    sample_n: int = CONFIG["judge_sample_n"],
    random_state: int = CONFIG["random_seed"],
) -> pd.DataFrame:
    if sample_n == 0:
        print("judge_sample_n = 0, skipping LLM judging.")
        return results_df.copy()

    df = results_df.copy()
    # Ensure judge columns exist
    for col in ["llm_correctness_score", "llm_correctness_reason",
                "llm_claims_covered",
                "llm_faithfulness_score", "llm_faithfulness_reason"]:
        if col not in df.columns:
            df[col] = None

    pipelines = df["pipeline"].unique()
    for pipeline in pipelines:
        pipe_idx = df[df["pipeline"] == pipeline].index.tolist()
        n = min(sample_n, len(pipe_idx))
        rng = random.Random(random_state)
        sampled_idx = rng.sample(pipe_idx, n)

        print(f"Judging {n} questions for [{pipeline}]...")
        for idx in tqdm(sampled_idx, desc=pipeline, leave=False):
            row = df.loc[idx]

            # Correctness: how right is the answer vs. the reference?
            c_score, c_claims, c_reason = llm_judge_correctness(
                row["question"],
                row["reference_answer"],
                row["final_answer"],
            )

            # Faithfulness: is the answer grounded in the retrieved context?
            retrieved_chunks = row.get("retrieved_chunks") if "retrieved_chunks" in df.columns else []
            if retrieved_chunks and isinstance(retrieved_chunks, list) and len(retrieved_chunks) > 0:
                context_str = format_retrieved_context(retrieved_chunks)
            else:
                context_str = row.get("final_answer", "")

            f_score, f_claims, f_reason = llm_judge_faithfulness(
                row["question"],
                context_str,
                row["final_answer"],
            )

            df.at[idx, "llm_correctness_score"]   = c_score
            df.at[idx, "llm_correctness_reason"]  = c_reason
            df.at[idx, "llm_claims_covered"]       = c_claims  # from correctness judge
            df.at[idx, "llm_faithfulness_score"]   = f_score
            df.at[idx, "llm_faithfulness_reason"]  = f_reason

            # Small delay between judge calls too
            time.sleep(CONFIG["inter_question_sleep_sec"])

    return df


results_df = run_llm_judging(results_df)
print("Done. Judge scores filled for sampled rows.")
results_df[["pipeline", "question", "final_answer", "llm_correctness_score", "llm_faithfulness_score"]].dropna(subset=["llm_correctness_score"]).head(12)


Judging 5 questions for [simple_rag]...


simple_rag:   0%|          | 0/5 [00:00<?, ?it/s]

  [WARN] JudgeOutput attempt 1/3 failed: RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499726, Requested 1199. Please try again in 2m39.84s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] JudgeOutput attempt 2/3 failed: RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499697, Requested 1199. Please try again in 2m34.8288s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] JudgeOutput attempt 3

advanced_rag:   0%|          | 0/5 [00:00<?, ?it/s]

  [WARN] JudgeOutput attempt 1/3 failed: RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499450, Requested 1224. Please try again in 1m56.4672s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] JudgeOutput attempt 2/3 failed: RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499421, Requested 1224. Please try again in 1m51.455999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] JudgeOutput at

agentic_rag_langgraph:   0%|          | 0/5 [00:00<?, ?it/s]

  [WARN] JudgeOutput attempt 1/3 failed: RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499813, Requested 1175. Please try again in 2m50.726399999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] JudgeOutput attempt 2/3 failed: RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499784, Requested 1175. Please try again in 2m45.7152s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] JudgeOutput at

,pipeline,question,final_answer,llm_correctness_score,llm_faithfulness_score
0,simple_rag,Does Corning have positive working capital bas...,To determine if Corning has positive working c...,0.0,0.0
1,advanced_rag,Does Corning have positive working capital bas...,To determine if Corning has positive working c...,0.0,0.0
2,agentic_rag_langgraph,Does Corning have positive working capital bas...,To determine if Corning has positive working c...,0.0,0.0
9,simple_rag,Has CVS Health paid dividends to common shareh...,,0.0,0.0
10,advanced_rag,Has CVS Health paid dividends to common shareh...,,0.0,0.0
11,agentic_rag_langgraph,Has CVS Health paid dividends to common shareh...,,0.0,0.0
24,simple_rag,What is Coca Cola's FY2021 COGS % margin? Calc...,To calculate the COGS (Cost of Goods Sold) % m...,0.0,0.0
25,advanced_rag,What is Coca Cola's FY2021 COGS % margin? Calc...,,0.0,0.0
26,agentic_rag_langgraph,What is Coca Cola's FY2021 COGS % margin? Calc...,To calculate the COGS (Cost of Goods Sold) % m...,0.0,0.0
60,simple_rag,How much was the Real change in Sales for AMCO...,The Real change in Sales for AMCOR in FY 2023 ...,0.0,0.0


## 18. Aggregate metrics

In [20]:
def aggregate_metrics(results_df: pd.DataFrame) -> pd.DataFrame:
    def nanmean_bool(series, match_val=None):
        if match_val is not None:
            vals = [v == match_val for v in series if pd.notna(v)]
        else:
            vals = [v for v in series if pd.notna(v)]
        return np.mean(vals) if vals else np.nan

    agg = (
        results_df.groupby("pipeline")
        .agg(
            n_questions              = ("id",                     "count"),
            exact_match              = ("exact_match",            "mean"),
            token_f1                 = ("token_f1",               "mean"),
            numerical_accuracy       = ("numerical_accuracy",     "mean"),
            decision_accuracy        = ("decision_accuracy",      "mean"),
            retrieval_hit_rate       = ("retrieval_hit",          "mean"),
            avg_latency_sec          = ("latency_sec",            "mean"),
            avg_context_retries      = ("context_retries",        "mean"),
            context_reject_rate      = ("context_eval_relevant",  lambda x: nanmean_bool([v is False for v in x if pd.notna(v)])),
            critic_repair_rate       = ("critic_decision",        lambda x: nanmean_bool(x, "repair")),
            repair_invocation_rate   = ("repair_used",            "mean"),
            repair_warn_rate         = ("repair_decision",        lambda x: nanmean_bool(x, "warn")),
            repair_refuse_rate       = ("repair_decision",        lambda x: nanmean_bool(x, "refuse")),
            avg_repair_retrievals    = ("repair_retrieval_count",  "mean"),
            llm_correctness_score    = ("llm_correctness_score",   lambda x: x.dropna().mean() if x.dropna().shape[0] > 0 else np.nan),
            llm_claims_covered       = ("llm_claims_covered",      lambda x: x.dropna().mean() if x.dropna().shape[0] > 0 else np.nan),
            llm_faithfulness_score   = ("llm_faithfulness_score",  lambda x: x.dropna().mean() if x.dropna().shape[0] > 0 else np.nan),
        )
        .reset_index()
    )
    return agg


summary_df = aggregate_metrics(results_df)
summary_df


,pipeline,n_questions,exact_match,token_f1,numerical_accuracy,decision_accuracy,retrieval_hit_rate,avg_latency_sec,avg_context_retries,context_reject_rate,critic_repair_rate,repair_invocation_rate,repair_warn_rate,repair_refuse_rate,avg_repair_retrievals,llm_correctness_score,llm_claims_covered,llm_faithfulness_score
0,advanced_rag,30,0.0,0.023198,0.300000,1.0,0.833333,19.498232,0.0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,0.0
1,agentic_rag_langgraph,30,0.0,0.031412,0.366667,1.0,0.833333,79.291294,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
2,simple_rag,30,0.0,0.025724,0.333333,1.0,0.833333,21.221566,0.0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,0.0


## 19. Repair success analysis

In [21]:
def compute_repair_success(results_df: pd.DataFrame) -> pd.DataFrame:
    agentic = results_df[results_df["pipeline"] == "agentic_rag_langgraph"].copy()
    if agentic.empty:
        return pd.DataFrame()

    repair_rows = agentic[agentic["repair_used"] == True].copy()
    if repair_rows.empty:
        return pd.DataFrame([{"pipeline": "agentic_rag_langgraph", "repair_success_rate": np.nan}])

    repair_success_rate = (repair_rows["token_f1"] >= 0.5).mean()
    return pd.DataFrame([{"pipeline": "agentic_rag_langgraph", "repair_success_rate": repair_success_rate}])


repair_success_df = compute_repair_success(results_df)
repair_success_df


,pipeline,repair_success_rate
0,agentic_rag_langgraph,NaN


## 20. Summary table

In [22]:
from IPython.display import display as ipy_display, Markdown

def display_markdown_summary(summary_df: pd.DataFrame, repair_success_df: pd.DataFrame):
    merged = summary_df.merge(repair_success_df, on="pipeline", how="left")

    # ── Key metrics summary table (renders as a proper HTML table in Jupyter) ──
    md  = "### Pipeline Evaluation Summary — FinanceBench\n\n"
    md += "| Pipeline | Num. Acc | Token F1 | Evidence Hit | LLM Correct | Claims Covered | LLM Faith | Avg Latency (s) |\n"
    md += "| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |\n"
    for _, r in merged.iterrows():
        corr   = f"{r['llm_correctness_score']:.3f}"  if pd.notna(r.get("llm_correctness_score"))  else "—"
        claims = f"{r['llm_claims_covered']:.3f}"     if pd.notna(r.get("llm_claims_covered"))      else "—"
        faith  = f"{r['llm_faithfulness_score']:.3f}" if pd.notna(r.get("llm_faithfulness_score")) else "—"
        md += (
            f"| {r['pipeline']} | {r['numerical_accuracy']:.3f} | {r['token_f1']:.3f} | "
            f"{r['retrieval_hit_rate']:.3f} | {corr} | {claims} | {faith} | {r['avg_latency_sec']:.2f} |\n"
        )
    ipy_display(Markdown(md))

    # ── Full metrics (all columns, styled) ──
    fmt = {
        "exact_match":           "{:.3f}",
        "token_f1":              "{:.3f}",
        "numerical_accuracy":    "{:.3f}",
        "decision_accuracy":     "{:.3f}",
        "retrieval_hit_rate":    "{:.3f}",
        "avg_latency_sec":       "{:.2f}",
        "avg_context_retries":   "{:.2f}",
        "context_reject_rate":   "{:.3f}",
        "critic_repair_rate":    "{:.3f}",
        "repair_invocation_rate":"{:.3f}",
        "repair_warn_rate":      "{:.3f}",
        "repair_refuse_rate":    "{:.3f}",
        "avg_repair_retrievals": "{:.2f}",
        "repair_success_rate":   "{:.3f}",
        "llm_correctness_score": "{:.3f}",
        "llm_claims_covered":    "{:.3f}",
        "llm_faithfulness_score":"{:.3f}",
    }
    ipy_display(merged.style.format(fmt, na_rep="—").set_caption("Full metrics (all pipelines)"))


display_markdown_summary(summary_df, repair_success_df)


### Pipeline Evaluation Summary — FinanceBench

| Pipeline | Num. Acc | Token F1 | Evidence Hit | LLM Correct | Claims Covered | LLM Faith | Avg Latency (s) |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| advanced_rag | 0.300 | 0.023 | 0.833 | 0.000 | 0.000 | 0.000 | 19.50 |
| agentic_rag_langgraph | 0.367 | 0.031 | 0.833 | 0.000 | 0.000 | 0.000 | 79.29 |
| simple_rag | 0.333 | 0.026 | 0.833 | 0.000 | 0.000 | 0.000 | 21.22 |


,pipeline,n_questions,exact_match,token_f1,numerical_accuracy,decision_accuracy,retrieval_hit_rate,avg_latency_sec,avg_context_retries,context_reject_rate,critic_repair_rate,repair_invocation_rate,repair_warn_rate,repair_refuse_rate,avg_repair_retrievals,llm_correctness_score,llm_claims_covered,llm_faithfulness_score,repair_success_rate
0,advanced_rag,30,0.000,0.023,0.300,1.000,0.833,19.50,0.00,—,—,0.000,—,—,0.00,0.000,0.000,0.000,—
1,agentic_rag_langgraph,30,0.000,0.031,0.367,1.000,0.833,79.29,0.00,0.000,0.000,0.000,—,—,0.00,0.000,0.000,0.000,—
2,simple_rag,30,0.000,0.026,0.333,1.000,0.833,21.22,0.00,—,—,0.000,—,—,0.00,0.000,0.000,0.000,—


## 21. Question-level inspection

In [23]:
cols_to_show = [
    "id", "pipeline", "company", "doc_name", "question",
    "reference_answer", "final_answer",
    "numerical_accuracy", "token_f1", "retrieval_hit",
    "llm_correctness_score", "llm_claims_covered", "llm_faithfulness_score",
    "critic_decision", "repair_used", "repair_decision", "repair_retrieval_count",
]
results_df[cols_to_show].head(12)


,id,pipeline,company,doc_name,question,reference_answer,final_answer,numerical_accuracy,token_f1,retrieval_hit,llm_correctness_score,llm_claims_covered,llm_faithfulness_score,critic_decision,repair_used,repair_decision,repair_retrieval_count
0,financebench_id_00005,simple_rag,Corning,CORNING_2022_10K,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,To determine if Corning has positive working c...,1.0,0.173913,1.0,0.0,0.0,0.0,NaN,False,None,0
1,financebench_id_00005,advanced_rag,Corning,CORNING_2022_10K,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,To determine if Corning has positive working c...,1.0,0.152047,1.0,0.0,0.0,0.0,NaN,False,None,0
2,financebench_id_00005,agentic_rag_langgraph,Corning,CORNING_2022_10K,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,To determine if Corning has positive working c...,1.0,0.171429,1.0,0.0,0.0,0.0,accept,False,None,0
3,financebench_id_06655,simple_rag,Amazon,AMAZON_2017_10K,What is Amazon's FY2017 days payable outstandi...,93.86,To calculate Amazon's FY2017 days payable outs...,0.0,0.000000,1.0,None,None,None,NaN,False,None,0
4,financebench_id_06655,advanced_rag,Amazon,AMAZON_2017_10K,What is Amazon's FY2017 days payable outstandi...,93.86,To calculate Amazon's FY2017 days payable outs...,0.0,0.000000,1.0,None,None,None,NaN,False,None,0
5,financebench_id_06655,agentic_rag_langgraph,Amazon,AMAZON_2017_10K,What is Amazon's FY2017 days payable outstandi...,93.86,To calculate Amazon's FY2017 days payable outs...,0.0,0.000000,1.0,None,None,None,accept,False,None,0
6,financebench_id_00080,simple_rag,Paypal,PAYPAL_2022_10K,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,To determine if Paypal has positive working ca...,1.0,0.119205,1.0,None,None,None,NaN,False,None,0
7,financebench_id_00080,advanced_rag,Paypal,PAYPAL_2022_10K,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,To determine if Paypal has positive working ca...,1.0,0.173913,1.0,None,None,None,NaN,False,None,0
8,financebench_id_00080,agentic_rag_langgraph,Paypal,PAYPAL_2022_10K,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,To determine if Paypal has positive working ca...,1.0,0.173913,1.0,None,None,None,accept,False,None,0
9,financebench_id_01244,simple_rag,CVS Health,CVSHEALTH_2022_10K,Has CVS Health paid dividends to common shareh...,"Yes, CVS paid a $ 0.55 dividend per share ever...",,0.0,0.000000,1.0,0.0,0.0,0.0,NaN,False,None,0


## 22. Save outputs if needed

In [24]:
# results_df.to_csv("question_level_results.csv", index=False)
# summary_df.to_csv("aggregate_metrics.csv", index=False)
# repair_success_df.to_csv("repair_success.csv", index=False)


## 23. Notes for adapting to your own SEC filings

The notebook is already running on FinanceBench, which uses real 10-K/10-Q text.
When you move from FinanceBench to your own filing set, the changes are minimal:

1. **Replace `load_financebench_subset`** with your own QA loader — keep the same output schema:
   `id, question, reference_answer, company, doc_name, question_type, gold_evidence_text, all_evidence_pages`

2. **Replace `get_filing_pages_for_question`** with a function that fetches/parses your filings.
   Options: `pdfplumber` for local PDFs, SEC EDGAR API for live fetching, or a pre-extracted text store.

3. **Chunking (`build_contextual_chunks`)** is already filing-aware — keep it as-is.
   If your filings have clear section structure (Item 1A, MD&A, Financial Statements),
   consider adding a section heading detection step to prefix each chunk with the section name.

4. **Retrieval hit metric** already uses gold evidence token overlap — works unchanged
   as long as your benchmark provides a `gold_evidence_text` field per question.

5. **Numerical accuracy** and **LLM judge prompts** are already tuned for financial answers — no changes needed.

6. **Shared index option**: if your question set covers fewer unique filings, consider building
   one shared `CorpusIndex` per filing rather than rebuilding per question.
   Sort `eval_df` by `doc_name` and cache the index between questions sharing a filing.


In [ ]:
# test